In [1]:
import os
import random
import librosa
import subprocess
import numpy as np

In [2]:
# Constants

SR = 41000  # sampling rate : number of numerical values per second of audio
CHUNK_DURATION = 5
CHUNK_SIZE = SR * CHUNK_DURATION  # total number of samples in a chunk
N_FFT = 2048  # number of samples in each FFT window
HOP_LENGTH = 512  # number of samples between successive frames
SPECTROGRAM_DIR_PATH = "../datasets/spectrograms/"
WAVEFORM_DIR_PATH = "../datasets/waveforms/"

GAIN_RANGE_DB = 6
TRAIN_MIX_FACTOR = 2    
aug_rng = random.Random(205)

train_dir = os.path.join(WAVEFORM_DIR_PATH, "train")
val_dir = os.path.join(WAVEFORM_DIR_PATH, "val")

os.makedirs(SPECTROGRAM_DIR_PATH, exist_ok=True)
os.makedirs(WAVEFORM_DIR_PATH, exist_ok=True)
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

In [3]:
def load_audio_from_mp4(path): #MUSDB dataset has audio file in mp4 format that's why we need use to ffmpeg else just librosa was fine 

    result = subprocess.run(
        [
            "ffmpeg",
            "-v", "error",
            "-i", path,
            "-f", "f32le",
            "-acodec", "pcm_f32le",
            "-ac", "1",
            "-ar", str(SR),
            "pipe:1"
        ],
        capture_output=True, check=True)

    audio = np.frombuffer(result.stdout, dtype=np.float32)

    return audio

In [4]:
def get_vocal_instrument_audio_MUSDB():
    
    audio_files = [os.path.join('../datasets/MUSDB18/stem_files', f) for f in os.listdir('../datasets/MUSDB18/stem_files')]
 
    vocal_files = {
        os.path.basename(f).replace('.stem_vocals.mp4', ''): f
        for f in audio_files if 'stem_vocals' in f
    }
 
    instrumental_files = {
        os.path.basename(f).replace('.stem_instrumental.mp4', ''): f
        for f in audio_files if 'stem_instrumental' in f
    }
 
    common_songs = sorted(vocal_files.keys() & instrumental_files.keys())
 
    vocal_audios = []
    instrument_audios = []
 
    for song in common_songs:
 
        vocals = load_audio_from_mp4(vocal_files[song])
        instrumental = load_audio_from_mp4(instrumental_files[song])
 
        length = min(len(vocals), len(instrumental))
 
        vocal_audios.append(vocals[:length])
        instrument_audios.append(instrumental[:length])
 
    return vocal_audios, instrument_audios

In [5]:
def get_chunks(signal):

    num_chunks = len(signal)//CHUNK_SIZE

    chunks = [signal[i*CHUNK_SIZE:(i+1)*CHUNK_SIZE] for i in range(num_chunks)]
    return chunks

def compute_spectrogram(signal):

    stft = librosa.stft(signal, n_fft=N_FFT, hop_length=HOP_LENGTH)
    magnitude = np.abs(stft)
    return magnitude

def db_to_linear(db):
    return 10.0 ** (db / 20.0)
 
 
def apply_random_gain(signal, max_db=GAIN_RANGE_DB, rng=aug_rng):
    gain_db = rng.uniform(-max_db, max_db)
    return signal * db_to_linear(gain_db)

In [6]:
def MUSDB_pipeline(val_ratio=0.2, train_mix_factor=TRAIN_MIX_FACTOR):
 
    vocal_audios, instrument_audios = get_vocal_instrument_audio_MUSDB()
    num_songs = len(vocal_audios)
    indices = list(range(num_songs))
    random.Random(42).shuffle(indices)
 
    val_count = int(num_songs * val_ratio)
    val_indices = set(indices[:val_count])
    train_indices = set(indices[val_count:])
 
    split_vocal_chunks = {"train": [], "val": []}
    split_instrument_chunks = {"train": [], "val": []}
 
    for song_idx, (vocal_audio, instrument_audio) in enumerate(zip(vocal_audios, instrument_audios)):
        split = "val" if song_idx in val_indices else "train"
        split_vocal_chunks[split].extend(get_chunks(vocal_audio))
        split_instrument_chunks[split].extend(get_chunks(instrument_audio))
 
    counters = {"train": 0, "val": 0}
 
    for split in ("train", "val"):
        out_dir = val_dir if split == "val" else train_dir
        vocal_chunks = split_vocal_chunks[split]
        instrument_chunks = split_instrument_chunks[split]
        n = min(len(vocal_chunks), len(instrument_chunks))
        mix_factor = train_mix_factor if split == "train" else 1
 
        for i in range(n):
            # same position pairing
            pairings = [(vocal_chunks[i], instrument_chunks[i])]
 
            # cross mixed pairings
            for _ in range(mix_factor - 1):
                j = aug_rng.randrange(len(instrument_chunks))
                pairings.append((vocal_chunks[i], instrument_chunks[j]))
 
            for vocal_chunk, instrument_chunk in pairings:
                if split == "train":
                    vocal_chunk = apply_random_gain(vocal_chunk)
                    instrument_chunk = apply_random_gain(instrument_chunk)
 
                mixture_chunk = vocal_chunk + instrument_chunk
 
                mixture_spectrogram = compute_spectrogram(mixture_chunk)
                vocal_spectrogram = compute_spectrogram(vocal_chunk)
                instrument_spectrogram = compute_spectrogram(instrument_chunk)
 
                idx = counters[split]
                np.save(f"{out_dir}/_MUSDB_{idx}_mixture.npy", mixture_spectrogram)
                np.save(f"{out_dir}/_MUSDB_{idx}_vocals.npy", vocal_spectrogram)
                np.save(f"{out_dir}/_MUSDB_{idx}_instrumental.npy", instrument_spectrogram)
                counters[split] += 1
 
    print(f"Songs: {len(train_indices)} train / {len(val_indices)} val")
    print(f"Chunks: {counters['train']} train / {counters['val']} val")

In [ ]:

def MUSDB_waveform_pipeline(val_ratio=0.2, train_mix_factor=TRAIN_MIX_FACTOR):

    vocal_audios, instrument_audios = get_vocal_instrument_audio_MUSDB()
    assert len(vocal_audios) == len(instrument_audios)

    num_songs = len(vocal_audios)

    indices = list(range(num_songs))
    random.Random(42).shuffle(indices)

    val_count = int(num_songs * val_ratio)

    val_indices = set(indices[:val_count])
    train_indices = set(indices[val_count:])

    split_vocal_chunks = {
        "train": [],
        "val": []
    }

    split_instrument_chunks = {
        "train": [],
        "val": []
    }

    for song_idx, (vocal_audio, instrument_audio) in enumerate(
        zip(vocal_audios, instrument_audios)
    ):

        split = "val" if song_idx in val_indices else "train"

        vocal_chunks = get_chunks(vocal_audio)
        instrument_chunks = get_chunks(instrument_audio)
        assert len(vocal_chunks) == len(instrument_chunks)

        split_vocal_chunks[split].extend(vocal_chunks)
        split_instrument_chunks[split].extend(instrument_chunks)

    counters = {
        "train": 0,
        "val": 0}

    aug_rng = random.Random(42)

    for split in ("train", "val"):

        out_dir = train_dir if split == "train" else val_dir

        vocal_chunks = split_vocal_chunks[split]
        instrument_chunks = split_instrument_chunks[split]

        n = min(len(vocal_chunks), len(instrument_chunks))

        mix_factor = train_mix_factor if split == "train" else 1

        for i in range(n):

            pairings = [
                (vocal_chunks[i], instrument_chunks[i])
            ]

            for _ in range(mix_factor - 1):

                j = aug_rng.randrange(len(instrument_chunks))
                pairings.append((vocal_chunks[i], instrument_chunks[j]))

            for vocal_chunk, instrument_chunk in pairings:

                vocal_chunk = np.asarray(vocal_chunk, dtype=np.float32)
                instrument_chunk = np.asarray(instrument_chunk, dtype=np.float32)

                if split == "train":
                    vocal_chunk = apply_random_gain(vocal_chunk)
                    instrument_chunk = apply_random_gain(instrument_chunk)

                mixture_chunk = vocal_chunk + instrument_chunk

                assert vocal_chunk.shape == instrument_chunk.shape
                assert mixture_chunk.shape == vocal_chunk.shape

                # SoundFile format (T, C) -> Demucs format (C, T)
                vocal_chunk = np.ascontiguousarray(vocal_chunk.T)
                instrument_chunk = np.ascontiguousarray(instrument_chunk.T)
                mixture_chunk = np.ascontiguousarray(mixture_chunk.T)

                idx = counters[split]

                np.save(f"{out_dir}/_MUSDB_{idx}_mixture.npy",mixture_chunk)
                np.save(f"{out_dir}/_MUSDB_{idx}_vocals.npy",vocal_chunk)
                np.save(f"{out_dir}/_MUSDB_{idx}_instrumental.npy",instrument_chunk)

                counters[split] += 1

    print(f"Songs: {len(train_indices)} train / {len(val_indices)} val")
    print(f"Chunks: {counters['train']} train / "
            f"{counters['val']} val")

In [8]:
MUSDB_waveform_pipeline()

Songs: 80 train / 19 val
Chunks: 6896 train / 981 val


In [9]:
file = os.listdir(train_dir)[0]
waveform = np.load(os.path.join(train_dir, file))

print("File:", file)
print("Shape:", waveform.shape)

File: _MUSDB_0_instrumental.npy
Shape: (205000,)
